# CIC-DDoS2019 Data Construction

### Setup

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os

def find_root_dir(marker='tfg'):
    """
    Find the project root directory
    """
    p = Path.cwd()
    for candidate in [p] + list(p.parents):
        if candidate.name == marker:
            return candidate.resolve()
        if (candidate / marker).is_dir():
            return (candidate / marker).resolve()
    raise FileNotFoundError(f"Could not find '{marker}' folder in {Path.cwd()} or its parents.")

## Data Construction

In [3]:
import pandas as pd

def create_interim_datasets(
    input_path: str | Path = "data/raw/CIC-DDoS2019.csv",
    output_dir: str | Path = "data/intermediate",
    sample_size_per_class: int = 4_000,
    random_state: int = 42,
) -> tuple[Path, Path]:
    """
    Create and save reference and full datasets as .parquet
    """

    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading dataset from: {input_path}")

    df = pd.read_csv(input_path)

    print(f"Full dataset shape: {df.shape}")
    print("\nFull label distribution:")
    print(df[" Label"].value_counts())

    # Create a balanced reference
    ddos_data = df[df[" Label"] == "DDoS"]
    benign_data = df[df[" Label"] == "BENIGN"]

    if len(ddos_data) < sample_size_per_class:
        raise ValueError(f"Not enough DDoS samples, available only: {len(ddos_data)})")
    if len(benign_data) < sample_size_per_class:
        raise ValueError(f"Not enough BENIGN samples, available only: {len(benign_data)})")

    ddos_samples = ddos_data.sample(
        n=sample_size_per_class,
        random_state=random_state,
    )

    benign_samples = benign_data.sample(
        n=sample_size_per_class,
        random_state=random_state,
    )

    reference_df = pd.concat(
        [ddos_samples, benign_samples],
        ignore_index=True,
    )

    reference_df = reference_df.sample(
        frac=1,
        random_state=random_state,
    ).reset_index(drop=True)

    reference_path = output_dir / "reference-8k.parquet"
    full_path = output_dir / "full-225k.parquet"
    reference_df.to_parquet(
        reference_path,
        index=False,
    )
    df.to_parquet(
        full_path,
        index=False,
    )

    print(f"\nReference dataset saved to: {reference_path}")
    print(f"Reference shape: {reference_df.shape}")
    print(reference_df[" Label"].value_counts())

    print(f"\nFull dataset saved to: {full_path}")
    print(f"Full shape: {df.shape}")

    return reference_path, full_path

In [4]:
root_dir = find_root_dir("tfg")

reference_path, full_path = create_interim_datasets(
    input_path=root_dir / "data/raw/CIC-DDoS2019.csv",
    output_dir=root_dir / "data/intermediate",
)

Loading dataset from: C:\Users\Monon\university\4th-year-bachelor\tfg\data\raw\CIC-DDoS2019.csv
Full dataset shape: (225745, 85)

Full label distribution:
 Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Reference dataset saved to: C:\Users\Monon\university\4th-year-bachelor\tfg\data\intermediate\reference-8k.parquet
Reference shape: (8000, 85)
 Label
DDoS      4000
BENIGN    4000
Name: count, dtype: int64

Full dataset saved to: C:\Users\Monon\university\4th-year-bachelor\tfg\data\intermediate\full-225k.parquet
Full shape: (225745, 85)
